# Diviser pour régner — Exercice 1 : les mots mystérieux

**Notions travaillées :** lecture de fichier, recherche **séquentielle** vs recherche **dichotomique**, coût d'un algorithme.

> **Énoncé (résumé)**
> `dico.txt` contient les mots de la langue française triés dans l'ordre lexicographique. `mots.txt` contient 20 000 mots, presque tous présents dans `dico.txt`.
> Afficher les quelques mots absents du dictionnaire **en moins de deux secondes**.

⚠️ Les fichiers `dico.txt` et `mots.txt` sont fournis par le professeur. S'ils sont placés **à côté de ce notebook**, ils sont utilisés. Sinon, le notebook fabrique automatiquement des **fichiers de démonstration** de même taille, avec 5 mots mystérieux, pour que tout fonctionne quand même.

In [1]:
import os, random, time

if not (os.path.exists("dico.txt") and os.path.exists("mots.txt")):
    print("Fichiers absents : création de fichiers de démonstration...")
    random.seed(1)
    lettres = "abcdefghijklmnopqrstuvwxyz"
    mysteres = ["dichotomiel", "fusionnade", "pythonnerie", "recursivement", "xylophonix"]
    ensemble = set()
    while len(ensemble) < 300000:
        ensemble.add("".join(random.choices(lettres, k=random.randint(3, 10))))
    ensemble -= set(mysteres)
    liste = sorted(ensemble)
    with open("dico.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(liste) + "\n")
    tirage = random.sample(liste, 20000 - len(mysteres)) + mysteres
    random.shuffle(tirage)
    with open("mots.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(tirage) + "\n")
print("Fichiers prêts.")

Fichiers absents : création de fichiers de démonstration...


Fichiers prêts.


## Question 1

### (a) Charger le dictionnaire

Le code de l'énoncé :

In [2]:
f = open("dico.txt", "r", encoding="utf-8")

dico = []
for ligne in f:
    dico.append(ligne[:-1])
f.close()

💡 **Explication ligne par ligne**
- `open(..., "r", encoding="utf-8")` ouvre le fichier en **lecture**. L'encodage UTF-8 est indispensable pour lire correctement les accents (é, è, ç...).
- `for ligne in f` parcourt le fichier **ligne par ligne**. Chaque ligne se termine par le caractère de retour à la ligne `"\n"`.
- `ligne[:-1]` enlève le **dernier caractère**, c'est-à-dire ce `"\n"`.

⚠️ **Piège :** si la dernière ligne du fichier ne se termine pas par `"\n"`, `ligne[:-1]` supprime la dernière **lettre** du dernier mot. La version plus sûre est `ligne.rstrip("\n")`, ou `ligne.strip()`.

### (b) Combien de mots ?

In [3]:
print(len(dico), "mots dans le dictionnaire")
print("Premiers mots :", dico[:5])

300000 mots dans le dictionnaire
Premiers mots : ['aaa', 'aaafs', 'aaafvyn', 'aaago', 'aaahsy']


## Question 2 : trouver les mots absents en moins de 2 secondes

### Première idée : la recherche séquentielle (`in` sur une liste)

`mot in dico` parcourt la liste **du début jusqu'à trouver le mot**. Pour un dictionnaire de $N$ mots, cela fait en moyenne $N/2$ comparaisons, et $N$ pour un mot absent.

Pour 20 000 mots et $N \approx 300\,000$ : environ $20\,000 \times 150\,000 = 3 \times 10^9$ comparaisons, **beaucoup trop**. On le vérifie en chronométrant seulement les **200 premiers mots**.

In [4]:
with open("mots.txt", encoding="utf-8") as f:
    mots = [ligne.rstrip("\n") for ligne in f]

debut = time.time()
for mot in mots[:200]:
    if mot not in dico:
        pass
duree = time.time() - debut
print(f"200 mots : {duree:.2f} s  ->  estimation pour {len(mots)} mots : {duree * len(mots) / 200:.0f} s")

200 mots : 0.35 s  ->  estimation pour 20000 mots : 35 s


### La bonne idée : la recherche dichotomique (« diviser pour régner »)

Le dictionnaire est **trié**. Pour chercher un mot :
1. on regarde le mot **au milieu** de la zone de recherche ;
2. s'il est égal au mot cherché, c'est gagné ;
3. s'il est **plus grand**, le mot cherché ne peut être que dans la **moitié gauche**, sinon dans la **moitié droite** ;
4. on recommence sur la moitié restante, jusqu'à ce que la zone soit vide.

À chaque étape, la zone de recherche est **divisée par 2**. Pour $N = 300\,000$, il faut au plus environ $\log_2(300\,000) \approx 19$ comparaisons, au lieu de 300 000 !

In [5]:
def recherche_dichotomique(t, x):
    # renvoie True si x est dans le tableau TRIÉ t
    gauche, droite = 0, len(t) - 1
    while gauche <= droite:
        milieu = (gauche + droite) // 2
        if t[milieu] == x:
            return True
        elif t[milieu] < x:
            gauche = milieu + 1      # x est forcément à droite du milieu
        else:
            droite = milieu - 1      # x est forcément à gauche du milieu
    return False


assert recherche_dichotomique([2, 5, 7, 9, 12], 9)
assert not recherche_dichotomique([2, 5, 7, 9, 12], 8)
assert not recherche_dichotomique([], 8)

⚠️ **Point essentiel : la dichotomie n'est correcte que si le tableau est trié dans le même ordre que celui de Python.**
Le fichier est trié « dans l'ordre lexicographique » de la langue française. Python, lui, compare les caractères selon leur code Unicode, et place par exemple `"é"` **après** `"z"`. Si les deux ordres diffèrent, la dichotomie peut « rater » des mots présents.
On vérifie donc d'abord avec une fonction `est_trie` (c'est aussi la question 1 de l'exercice 2) et, si besoin, on **trie une seule fois** le dictionnaire avec `sorted`.

In [6]:
def est_trie(t):
    for i in range(len(t) - 1):
        if t[i] > t[i + 1]:
            return False
    return True


debut = time.time()

if not est_trie(dico):
    print("Le dictionnaire n'est pas trié dans l'ordre de Python : on le trie.")
    dico = sorted(dico)

mysteres = [mot for mot in mots if not recherche_dichotomique(dico, mot)]

print("Mots mystérieux :", mysteres)
print(f"Temps total : {time.time() - debut:.3f} s")

Mots mystérieux : ['dichotomiel', 'pythonnerie', 'xylophonix', 'fusionnade', 'recursivement']
Temps total : 0.076 s


✅ Moins de 2 secondes : le contrat est rempli.

💡 **Comparaison des coûts**

| Méthode | Comparaisons par mot | Total pour 20 000 mots |
|:---|:---|:---|
| Recherche séquentielle | $\approx N$ ≈ 300 000 | ≈ 6 milliards |
| Recherche dichotomique | $\approx \log_2(N)$ ≈ 19 | ≈ 380 000 |

**Pour aller plus loin :** le module `bisect` de Python fournit la dichotomie toute faite, et un `set` (ensemble) permet de tester l'appartenance en temps quasi constant : `set(dico)`. Ce n'est pas l'objet de l'exercice, qui demande d'utiliser l'idée « diviser pour régner ».